### Построение лагов и rolling features без leakage

In [1]:
import pandas as pd
import numpy as np

In [2]:
DATE_COL = "rep_date"
COUNTRY_COL = "country"
HS_COL = "hs"
TARGET_COL = "value"

STATIC_COUNTRY_COLS = [
    "distw",
    "unfriendly",
    "brics",
    "cis"
]

COUNTRY_TIME_COLS = [
    "sanctions_proxy",
    "sanctions_proxy_smooth",
    "cpi_yoy",
    "ip_yoy",
    "ex_yoy",
    "logistics_exposure_distw"
]

GLOBAL_TIME_COLS = [
    "gscpi",
    "cb_full_import_value",
    "usd_rub",
    "gov_earn_from_import_nds",
    "gov_expendings"
]

OPTIONAL_EXISTING_COLS = [
    "post_sanctions",
    "unfriendly_post",
    "brics_post",
    "cis_post"
]

SANCTIONS_START = pd.Timestamp("2022-02-01")

TARGET_LAGS = [1, 2, 3, 6, 12]
TARGET_ROLL_WINDOWS = [3, 6, 12]

DYNAMIC_LAGS = [1, 3, 6, 12]
DYNAMIC_ROLL_WINDOWS = [3, 6]

MIN_HISTORY_STRICT = 12

#### Безопасная перестройка санкционного smoothing

In [3]:
def rebuild_safe_sanctions_features(
    df: pd.DataFrame,
    date_col: str = DATE_COL,
    country_col: str = COUNTRY_COL,
    raw_col: str = "sanctions_proxy"
) -> pd.DataFrame:
    """
    Создает безопасную trailing-версию сглаженного санкционного прокси.
    Использует только информацию, доступную на момент t.
    """
    data = df.copy()
    data = data.sort_values([country_col, date_col]).reset_index(drop=True)

    data["sanctions_proxy_ma3_trailing"] = (
        data.groupby(country_col)[raw_col]
            .transform(lambda s: s.rolling(window=3, min_periods=1).mean())
    )

    data["sanctions_proxy_ma6_trailing"] = (
        data.groupby(country_col)[raw_col]
            .transform(lambda s: s.rolling(window=6, min_periods=1).mean())
    )

    return data

#### Лаги и rolling features для target-ряда

In [4]:
def add_target_history_features(
    df: pd.DataFrame,
    series_col: str = "series_id",
    date_col: str = DATE_COL,
    target_col: str = TARGET_COL,
    lags: list = TARGET_LAGS,
    roll_windows: list = TARGET_ROLL_WINDOWS
) -> pd.DataFrame:
    """
    Строит лаги и rolling features для target-ряда на уровне country × hs.
    Используется только история до момента t.
    """
    data = df.copy()
    data = data.sort_values([series_col, date_col]).reset_index(drop=True)

    g = data.groupby(series_col)[target_col]

    # Лаги
    for lag in lags:
        data[f"{target_col}_lag{lag}"] = g.shift(lag)

    # Последнее известное значение
    data[f"{target_col}_last_observed"] = data[target_col]

    # Rolling mean / std / median / max по истории до t включительно
    for w in roll_windows:
        data[f"{target_col}_rollmean_{w}"] = (
            g.transform(lambda s: s.rolling(window=w, min_periods=1).mean())
        )
        data[f"{target_col}_rollstd_{w}"] = (
            g.transform(lambda s: s.rolling(window=w, min_periods=2).std())
        )
        data[f"{target_col}_rollmedian_{w}"] = (
            g.transform(lambda s: s.rolling(window=w, min_periods=1).median())
        )
        data[f"{target_col}_rollmax_{w}"] = (
            g.transform(lambda s: s.rolling(window=w, min_periods=1).max())
        )

    # Доля нулей в последних окнах
    zero_ind = (data[target_col] == 0).astype(int)
    data["_zero_ind_tmp"] = zero_ind

    gz = data.groupby(series_col)["_zero_ind_tmp"]
    for w in roll_windows:
        data[f"{target_col}_zero_share_{w}"] = (
            gz.transform(lambda s: s.rolling(window=w, min_periods=1).mean())
        )

    # Признак нулевого текущего значения
    data[f"{target_col}_is_zero_now"] = (data[target_col] == 0).astype(int)

    # MOM / YOY динамика
    data[f"{target_col}_mom_ratio"] = data[target_col] / (data[f"{target_col}_lag1"] + 1e-9)
    data[f"{target_col}_yoy_ratio"] = data[target_col] / (data[f"{target_col}_lag12"] + 1e-9)

    data[f"{target_col}_mom_diff"] = data[target_col] - data[f"{target_col}_lag1"]
    data[f"{target_col}_yoy_diff"] = data[target_col] - data[f"{target_col}_lag12"]

    # Лог-версия как дополнительный стабильный признак
    data[f"log1p_{target_col}"] = np.log1p(data[target_col])

    # Очистка
    data = data.drop(columns=["_zero_ind_tmp"])

    return data

#### Универсальная функция для динамических признаков

In [5]:
def add_dynamic_history_features(
    base_df: pd.DataFrame,
    group_cols: list,
    value_cols: list,
    date_col: str = DATE_COL,
    lags: list = DYNAMIC_LAGS,
    roll_windows: list = DYNAMIC_ROLL_WINDOWS,
    suffix_prefix: str = ""
) -> pd.DataFrame:
    """
    На вход получает dataframe на нужном логическом уровне:
    - country-time: group_cols = [country]
    - global-time:  group_cols = []
    
    Возвращает dataframe с лагами и rolling features.
    """
    data = base_df.copy()
    sort_cols = group_cols + [date_col] if group_cols else [date_col]
    data = data.sort_values(sort_cols).reset_index(drop=True)

    if group_cols:
        grouped = data.groupby(group_cols)
    else:
        grouped = None

    for col in value_cols:
        # Лаги
        for lag in lags:
            new_col = f"{suffix_prefix}{col}_lag{lag}"
            if group_cols:
                data[new_col] = grouped[col].shift(lag)
            else:
                data[new_col] = data[col].shift(lag)

        # Rolling mean / std
        for w in roll_windows:
            mean_col = f"{suffix_prefix}{col}_rollmean_{w}"
            std_col = f"{suffix_prefix}{col}_rollstd_{w}"

            if group_cols:
                data[mean_col] = grouped[col].transform(
                    lambda s: s.rolling(window=w, min_periods=1).mean()
                )
                data[std_col] = grouped[col].transform(
                    lambda s: s.rolling(window=w, min_periods=2).std()
                )
            else:
                data[mean_col] = data[col].rolling(window=w, min_periods=1).mean()
                data[std_col] = data[col].rolling(window=w, min_periods=2).std()

        # MOM / YOY differences
        lag1_col = f"{suffix_prefix}{col}_lag1"
        lag12_col = f"{suffix_prefix}{col}_lag12"

        data[f"{suffix_prefix}{col}_mom_diff"] = data[col] - data[lag1_col]
        data[f"{suffix_prefix}{col}_yoy_diff"] = data[col] - data[lag12_col]

    return data

#### Country-time features

In [6]:
def build_country_time_feature_block(
    df: pd.DataFrame,
    date_col: str = DATE_COL,
    country_col: str = COUNTRY_COL
) -> pd.DataFrame:
    """
    Строит лаги и rolling features для country-time переменных.
    """
    data = df.copy()

    country_time_base_cols = [
        "sanctions_proxy",
        "cpi_yoy",
        "ip_yoy",
        "ex_yoy",
        "logistics_exposure_distw"
    ]

    # Если safe smoothing уже перестроен, используем именно его
    if "sanctions_proxy_ma3_trailing" in data.columns:
        country_time_base_cols.append("sanctions_proxy_ma3_trailing")

    if "sanctions_proxy_ma6_trailing" in data.columns:
        country_time_base_cols.append("sanctions_proxy_ma6_trailing")

    country_time_df = (
        data.groupby([date_col, country_col], as_index=False)[country_time_base_cols]
            .agg("first")
    )

    country_time_df = add_dynamic_history_features(
        base_df=country_time_df,
        group_cols=[country_col],
        value_cols=country_time_base_cols,
        date_col=date_col,
        lags=DYNAMIC_LAGS,
        roll_windows=DYNAMIC_ROLL_WINDOWS,
        suffix_prefix="ct_"
    )

    return country_time_df

#### Global-time features

In [7]:
def build_global_time_feature_block(
    df: pd.DataFrame,
    date_col: str = DATE_COL
) -> pd.DataFrame:
    """
    Строит лаги и rolling features для global time-only переменных.
    """
    data = df.copy()

    global_time_base_cols = [
        "gscpi",
        "cb_full_import_value",
        "usd_rub",
        "gov_earn_from_import_nds",
        "gov_expendings"
    ]

    global_time_df = (
        data.groupby(date_col, as_index=False)[global_time_base_cols]
            .agg("first")
    )

    global_time_df = add_dynamic_history_features(
        base_df=global_time_df,
        group_cols=[],
        value_cols=global_time_base_cols,
        date_col=date_col,
        lags=DYNAMIC_LAGS,
        roll_windows=DYNAMIC_ROLL_WINDOWS,
        suffix_prefix="gt_"
    )

    return global_time_df

#### Сборка полной modeling table

In [8]:
def build_modeling_table_block2(
    panel_h: pd.DataFrame,
    horizon: int = 1,
    date_col: str = DATE_COL,
    country_col: str = COUNTRY_COL,
    hs_col: str = HS_COL,
    target_col: str = TARGET_COL
) -> pd.DataFrame:
    """
    Полный pipeline второго блока:
    1) safe sanctions features
    2) target history features
    3) country-time dynamic features
    4) global-time dynamic features
    5) merge all into one modeling table
    """
    data = panel_h.copy()

    # 1. Безопасное сглаживание санкционного прокси
    data = rebuild_safe_sanctions_features(data)

    # 2. История target на уровне series_id
    data = add_target_history_features(data)

    # 3. Country-time block
    ct_block = build_country_time_feature_block(data)

    # 4. Global-time block
    gt_block = build_global_time_feature_block(data)

    # 5. Merge back
    out = data.merge(
        ct_block,
        on=[date_col, country_col],
        how="left",
        suffixes=("", "_drop_ct")
    )

    out = out.merge(
        gt_block,
        on=[date_col],
        how="left",
        suffixes=("", "_drop_gt")
    )

    # remove accidental duplicated columns
    drop_cols = [c for c in out.columns if c.endswith("_drop_ct") or c.endswith("_drop_gt")]
    if drop_cols:
        out = out.drop(columns=drop_cols)

    # usable flag for current horizon
    out[f"is_usable_h{horizon}"] = out[f"target_h{horizon}"].notna().astype(int)

    return out

#### Отбор modeling sample


In [9]:
def select_modeling_sample(
    df: pd.DataFrame,
    horizon: int = 1,
    mode: str = "strict",
    min_history: int = MIN_HISTORY_STRICT
) -> pd.DataFrame:
    """
    Отбирает строки для обучения.
    mode='strict' -> требует сезонную историю
    mode='soft'   -> только существование target
    """
    data = df.copy()

    usable_col = f"is_usable_h{horizon}"
    data = data[data[usable_col] == 1].copy()

    if mode == "strict":
        required_cols = [
            f"{TARGET_COL}_lag12",
            "ct_sanctions_proxy_lag12",
            "gt_gscpi_lag12",
            "gt_usd_rub_lag12"
        ]

        required_cols = [c for c in required_cols if c in data.columns]
        for col in required_cols:
            data = data[data[col].notna()]

    data = data.reset_index(drop=True)
    return data

#### Диагностика второго блока

In [10]:
def diagnostic_summary_block2(
    df: pd.DataFrame,
    horizon: int = 1,
    target_col: str = TARGET_COL
):
    print("=" * 90)
    print("BLOCK 2 DIAGNOSTIC SUMMARY")
    print("=" * 90)

    print(f"Rows total: {len(df):,}")
    print(f"Rows usable for h={horizon}: {df[f'is_usable_h{horizon}'].sum():,}")
    print(f"Target zero share (future): {(df[f'target_h{horizon}'] == 0).mean():.3f}")

    # ключевые группы признаков
    target_feats = [c for c in df.columns if c.startswith(f"{target_col}_lag") or c.startswith(f"{target_col}_roll")]
    ct_feats = [c for c in df.columns if c.startswith("ct_")]
    gt_feats = [c for c in df.columns if c.startswith("gt_")]

    print(f"\nTarget history features: {len(target_feats)}")
    print(f"Country-time dynamic features: {len(ct_feats)}")
    print(f"Global-time dynamic features: {len(gt_feats)}")

    # missingness
    key_check = []
    for c in [
        f"{target_col}_lag1",
        f"{target_col}_lag12",
        "ct_sanctions_proxy_lag1",
        "ct_sanctions_proxy_lag12",
        "gt_gscpi_lag1",
        "gt_gscpi_lag12",
        "gt_usd_rub_lag1",
        "gt_usd_rub_lag12"
    ]:
        if c in df.columns:
            key_check.append(c)

    if key_check:
        miss = df[key_check].isna().mean().sort_values()
        print("\nMissing share in key features:")
        print(miss)

In [11]:
panel_h1 = pd.read_excel('data/prepared_forecasting_data.xlsx')

model_df_h1 = build_modeling_table_block2(panel_h1, horizon=1)

diagnostic_summary_block2(model_df_h1, horizon=1)

# строгая выборка для дальнейшего моделирования
model_df_h1_strict = select_modeling_sample(model_df_h1, horizon=1, mode="strict")

print("\nStrict sample shape:", model_df_h1_strict.shape)
model_df_h1_strict.head()

BLOCK 2 DIAGNOSTIC SUMMARY
Rows total: 48,438
Rows usable for h=1: 47,817
Target zero share (future): 0.547

Target history features: 17
Country-time dynamic features: 70
Global-time dynamic features: 50

Missing share in key features:
value_lag1                  0.012821
ct_sanctions_proxy_lag1     0.012821
gt_gscpi_lag1               0.012821
gt_usd_rub_lag1             0.012821
value_lag12                 0.153846
ct_sanctions_proxy_lag12    0.153846
gt_gscpi_lag12              0.153846
gt_usd_rub_lag12            0.153846
dtype: float64

Strict sample shape: (40365, 184)


,rep_date,country,hs,value,sanctions_proxy,sanctions_proxy_smooth,cpi_yoy,ip_yoy,ex_yoy,gscpi,...,gt_gov_expendings_lag1,gt_gov_expendings_lag3,gt_gov_expendings_lag6,gt_gov_expendings_lag12,gt_gov_expendings_rollmean_3,gt_gov_expendings_rollstd_3,gt_gov_expendings_rollmean_6,gt_gov_expendings_rollstd_6,gt_gov_expendings_mom_diff,gt_gov_expendings_yoy_diff
0,2020-01-01,Albania,9018,0.0,0,0,1.451977,-0.8,0.873605,0.055359,...,5.224931e+10,2.434201e+10,2.318892e+10,1.683345e+10,3.354079e+10,1.633664e+10,2.753926e+10,1.235527e+10,-2.597011e+10,9.445742e+09
1,2020-02-01,Albania,9018,0.0,0,0,1.219155,-7.0,2.046768,1.302174,...,2.627919e+10,2.209387e+10,1.924515e+10,2.044704e+10,3.325028e+10,1.664677e+10,2.786879e+10,1.211382e+10,-5.056842e+09,7.753125e+08
2,2020-03-01,Albania,9018,0.0,0,0,2.113296,-13.2,1.586344,2.671692,...,2.122235e+10,5.224931e+10,2.102604e+10,2.241134e+10,2.323028e+10,2.684335e+09,2.806267e+10,1.199108e+10,9.669410e+08,-2.220413e+08
3,2020-04-01,Albania,9018,0.0,0,0,1.932130,-16.7,4.207182,3.385323,...,2.218929e+10,2.627919e+10,2.434201e+10,2.509582e+10,2.400008e+10,4.003073e+09,2.877043e+10,1.185207e+10,6.399301e+09,3.492776e+09
4,2020-05-01,Albania,9018,0.0,0,0,2.090558,-20.2,3.481882,2.576995,...,2.858860e+10,2.122235e+10,2.209387e+10,1.602645e+10,2.340595e+10,4.694095e+09,2.832812e+10,1.219562e+10,-9.148624e+09,3.413526e+09


In [15]:
model_df_h1_strict.to_excel('data/after_clean_lags_and_rolling_data.xlsx', index=False)

На втором этапе была сформирована система лаговых и агрегированных признаков, предназначенных для реалистичного прогнозирования без утечки информации из будущего. Поскольку фактические значения динамических country-time и global time-only переменных в прогнозируемом периоде неизвестны, они были преобразованы в набор исторических характеристик, доступных на момент построения прогноза.

Для зависимой переменной, отражающей объем импорта в денежном выражении на уровне пары «страна × HS4», были построены лаговые признаки на горизонтах 1, 2, 3, 6 и 12 месяцев, а также rolling statistics по последним 3, 6 и 12 месяцам, включая среднее, стандартное отклонение, медиану, максимум и долю нулевых наблюдений. Дополнительно были сформированы признаки текущей динамики, в том числе month-over-month и year-over-year differences. Такой набор позволяет модели учитывать инерцию импорта, сезонный паттерн и различия между стабильными и прерывистыми торговыми потоками.

Для country-time переменных, включая санкционный прокси, макроэкономические показатели стран-экспортеров и индекс логистического воздействия, были построены лаги и rolling means на уровне пары «месяц × страна», после чего они были присоединены обратно к основной панели. Отдельно для санкционного прокси была сформирована безопасная trailing moving average, использующая только текущие и прошлые значения, поскольку centered smoothing приводил бы к использованию будущей информации.

Для global time-only переменных, одинаковых для всех наблюдений внутри месяца, включая глобальный индекс логистического давления, совокупный импорт Российской Федерации, курс доллара к рублю, бюджетные доходы от импортного НДС и государственные расходы, были построены лаговые и rolling признаки на уровне общего временного ряда. В результате была получена единая матрица признаков для прогнозирования, содержащая как статические и календарные характеристики, так и историю торговых, санкционных, макроэкономических и общесистемных факторов.